# 06 — Inspect a finished run

Load `history.json`, plot the learning curve, evaluate the saved model on the validation set, and look at the embedding scatter (if `--dump-embeddings` was used).


In [ ]:
import os, json, numpy as np, matplotlib.pyplot as plt, sys
ROOT = os.path.dirname(os.path.abspath('.')) if os.path.basename(os.getcwd())=='notebooks' else os.path.abspath('.')
SRC = os.path.join(ROOT, 'src')
if SRC not in sys.path: sys.path.insert(0, SRC)

RUN_NAME = 'unet_full'                 # which run dir to inspect (under outputs/)
OUT = os.path.join(ROOT, 'outputs', RUN_NAME)
print('inspecting', OUT)


## Learning curve


In [ ]:
h = json.load(open(os.path.join(OUT, 'history.json')))['history']
steps = [r['step'] for r in h]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(steps, [r['train_loss'] for r in h], 'o-', ms=3, label='train')
a1.plot(steps, [r['val_loss'] for r in h], 's-', ms=3, label='val')
a1.set_yscale('log'); a1.legend(); a1.set_xlabel('file # (incremental step)'); a1.set_title('loss')
a2.plot(steps, [r['val_masked_mse'] for r in h], 's-', ms=3, color='C2')
a2.set_yscale('log'); a2.set_xlabel('file # (incremental step)'); a2.set_title('val masked MSE (occluded region)')
plt.tight_layout(); plt.show()


## Final metrics vs. baselines


In [ ]:
print(json.dumps(json.load(open(os.path.join(OUT, 'final_metrics.json'))), indent=2))


## Embeddings (if --dump-embeddings was used)


In [ ]:
emb_path = os.path.join(OUT, 'embeddings.npy')
if os.path.exists(emb_path):
    emb = np.load(emb_path)
    print('embeddings:', emb.shape)
    Xc = emb - emb.mean(0, keepdims=True)
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    Z = Xc @ Vt[:2].T
    plt.figure(figsize=(6,5)); sc = plt.scatter(Z[:,0], Z[:,1], c=np.arange(len(Z)), cmap='viridis', s=8)
    plt.colorbar(sc, label='val sample index (time order)'); plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title('embeddings — 2-D PCA'); plt.show()
else:
    print('no embeddings.npy — run was without --dump-embeddings')
